# Chapter 2 assignment: a posteriori error bounds for PINNs

**Book:** *Physics-Informed Neural Networks for Ordinary, Integral and Integro-Differential Equations*, Chapter 2 (Approximation, optimization and error analysis), Section 2.5.

## Learning goals
After this assignment you will be able to

1. evaluate the residual and the initial defect of a trained PINN on a fine grid,
2. implement the residual-based error bounds for initial value problems of Section 2.5 (one-sided and Lipschitz versions) with the trapezoidal rule,
3. verify that the bounds hold and measure how sharp they are (effectivity index), including a time-dependent one-sided constant, and
4. follow the effectivity of the bound during training.

Everything runs on a CPU in double precision; each training takes well under a minute.

## Short theory recap

For $y'=f(t,y)$, $y(0)=y_0$, a trial function $y_\theta$ has residual $r(t)=y_\theta'(t)-f(t,y_\theta(t))$ and initial defect $e_0=y_\theta(0)-y_0$.
If $\langle f(t,y_\theta)-f(t,y),\,y_\theta-y\rangle\le\mu(t)\,|y_\theta-y|^2$, then with $M(t)=\int_0^t\mu$
$$|y_\theta(t)-y(t)|\le e^{M(t)}|e_0|+\int_0^t e^{M(t)-M(s)}|r(s)|\,ds=:B_\mu(t).$$
A Lipschitz constant $L$ always gives $\mu=L$ (the *crude* bound). For $f(t,y)=-y+\sin t$ we have $L=1$ but $\mu=-1$.
For the logistic equation $y'=y(1-y)$ with $y(0)=y_0\in(0,1)$ one can use $\mu(t)=1-y_0-y_\theta(t)$.

The **effectivity index** is $\max_t B(t)/\max_t|y_\theta(t)-y(t)|\ (\ge 1)$.

In [ ]:
import numpy as np
import torch

from pinnode import MLP, configure_torch, derivative, set_seed, to_tensor, train

configure_torch("float64", threads=1)
set_seed(1234)

def cumtrapz(f, t):
    """Cumulative trapezoidal rule, starting with 0."""
    return np.concatenate([[0.0], np.cumsum(0.5 * (f[1:] + f[:-1]) * np.diff(t))])

## Task 1: train a PINN for $y'=-y+\sin t$, $y(0)=1$ on $[0,6]$

The exact solution is $y(t)=1.5e^{-t}+(\sin t-\cos t)/2$. The training cell is complete; run it.

In [ ]:
T, Y0 = 6.0, 1.0
exact = lambda t: 1.5 * np.exp(-t) + 0.5 * (np.sin(t) - np.cos(t))
torch.manual_seed(1234)
model = MLP(1, 1, width=32, depth=3, activation="tanh", domain=(0.0, T))
t_col = to_tensor(np.linspace(0.0, T, 120), requires_grad=True)
t0 = to_tensor([0.0])

def loss_fn():
    y = model(t_col)
    r = derivative(y, t_col) + y - torch.sin(t_col)
    L_r, L_ic = (r ** 2).mean(), ((model(t0) - Y0) ** 2).sum()
    return L_r + L_ic, {"residual": L_r, "ic": L_ic}

hist = train(model.parameters(), loss_fn, adam_iters=2000, lr=1e-3, lr_final=1e-5,
             lbfgs_iters=300, print_every=1000)

## Task 2: residual on a fine grid and the two bounds

1. Evaluate $y_\theta$, $|r|$ and $|e_0|$ on 2001 uniform points (use `derivative`).
2. Implement `bound(t, r_abs, e0_abs, mu)` for a constant **or** an array `mu` (use `cumtrapz` for $M$ and for the integral).
3. Compute the sharp bound ($\mu=-1$), the crude bound ($\mu=L=1$) and the true error.

In [ ]:
t_f = np.linspace(0.0, T, 2001)

def residual_on_grid(model, t, rhs):
    tt = to_tensor(t, requires_grad=True)
    y = model(tt)
    r = derivative(y, tt) - rhs(tt, y)
    return y.detach().numpy().ravel(), np.abs(r.detach().numpy().ravel())

y_f, r_abs = residual_on_grid(model, t_f, lambda t, y: -y + torch.sin(t))
with torch.no_grad():
    e0_abs = abs(model(t0).item() - Y0)

def bound(t, r_abs, e0_abs, mu):
    mu = np.broadcast_to(np.asarray(mu, float), t.shape)
    # TODO: compute M = cumtrapz(mu, t) and return exp(M) * (e0_abs + cumtrapz(exp(-M) * r_abs, t))
    raise NotImplementedError

err = np.abs(y_f - exact(t_f))
B_sharp = bound(t_f, r_abs, e0_abs, -1.0)
B_crude = bound(t_f, r_abs, e0_abs, 1.0)

**Check your answer.** Both bounds must dominate the error at every grid point (up to the tiny quadrature error of the trapezoidal rule), and the sharp bound should be within a small factor of the error.

In [ ]:
assert np.all(B_sharp >= err * (1 - 1e-3)), "the sharp bound must not be smaller than the error"
assert np.all(B_crude >= B_sharp * (1 - 1e-12)), "the crude bound is never smaller than the sharp one"
print(f"max error {err.max():.2e}, sharp bound {B_sharp.max():.2e} (effectivity {B_sharp.max() / err.max():.2f}), "
      f"crude bound {B_crude.max():.2e}")
print("Task 2 passed.")

## Task 3: a time-dependent one-sided constant (logistic equation)

Train a PINN for $y'=y(1-y)$, $y(0)=0.1$ on $[0,5]$ (exact solution $y=0.1e^t/(1+0.1(e^t-1))$) with the settings of Task 1,
evaluate the bound with $\mu(t)=0.9-y_\theta(t)$ and with the Lipschitz constant $L=\max|1-2s|$ over the range of $y_\theta$ and $y$, and check validity.
Where is $\mu(t)$ positive, and how does this show in the bound?

In [ ]:
# TODO: train the logistic PINN (copy and adapt the cell of Task 1),
#       evaluate y_f, r_abs, e0_abs with residual_on_grid(..., lambda t, y: y * (1 - y)),
#       compute B = bound(t_f, r_abs, e0_abs, 0.9 - y_f) and check np.all(B >= err * (1 - 1e-3))


## Task 4: effectivity during training

`train` accepts a callback `weight_update(k)` that is called before every Adam step. Use it to record, every 250 iterations, the maximum error and the
maximum of the sharp bound for the problem of Task 1 (retrain from a fresh initialization). Plot both against the iteration number on a logarithmic axis
and comment on the effectivity index: does the bound become more or less sharp as training proceeds? (Compare with the table of effectivity indices in Section 2.5 of the book.)

In [ ]:
records = []

def monitor(k):
    if k % 250 == 0:
        # TODO: evaluate residual_on_grid, the sharp bound and the error for the current model,
        #       and append (k, err.max(), B.max()) to records
        pass

# TODO: re-create the model, call train(..., weight_update=monitor), then plot the records


## Discussion questions
* Why does the crude bound grow like $e^{t}$ although the true error does not?
* The sharp bound for Task 2 is almost an equality on intervals where $r$ has constant sign. Explain this with the variation-of-constants formula.
* What would the bound report if the training had converged to a wrong solution? (See problem B$'$ in the table of Section 2.5.)